<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/transformers-llm/08_kuantizasyon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kuantizasyon (Quantization): FP16 → INT8 → INT4

Bir 7B parametreli model FP16'da **~14 GB** tutar; INT4'te **~3.5 GB**. Kuantizasyon, ağırlıkları (ve bazen aktivasyonları) daha az bitle temsil ederek:
- **Bellek** ihtiyacını düşürür → daha büyük model, daha ucuz donanım
- **Hızı** artırır → LLM üretimi bellek bant genişliğiyle sınırlıdır, daha az bayt = daha hızlı okuma

Bu defterde:
1. Sayı formatları: FP32, FP16, BF16, INT8, INT4
2. Bellek okuma hızı ve "token/sn" tavanı
3. **Simetrik** ve **asimetrik (zero-point)** kuantizasyonu sıfırdan yazmak
4. Aykırı değerler ve **per-channel / group-wise** kuantizasyon
5. **PTQ** (Post-Training Quantization): eğitilmiş modeli sonradan kuantize etmek
6. **QAT** (Quantization-Aware Training): Straight-Through Estimator ile kuantizasyonu eğitime dahil etmek

In [ ]:
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 100
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

---
## 1) Sayı formatları

| Format | Bit düzeni (işaret–üs–mantis) | Aralık | Not |
|---|---|---|---|
| FP32 | 1 – 8 – 23 | ±3.4·10³⁸ | Eğitimde varsayılan |
| FP16 | 1 – 5 – 10 | ±65 504 | Dar aralık → taşma (overflow) riski |
| BF16 | 1 – 8 – 7 | ±3.4·10³⁸ | FP32 ile aynı aralık, daha az hassasiyet; modern eğitim standardı |
| INT8 | 8 bit tam sayı | −128 … 127 | 256 seviye |
| INT4 | 4 bit tam sayı | −8 … 7 | Sadece **16** seviye! |

In [ ]:
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    bilgi = torch.finfo(dtype)
    print(f"{str(dtype):15s} max = {bilgi.max:.3e}   en küçük adım (eps) = {bilgi.eps:.3e}")
for dtype in [torch.int8]:
    bilgi = torch.iinfo(dtype)
    print(f"{str(dtype):15s} min = {bilgi.min}, max = {bilgi.max}")

print("\nFP16 taşma  :", torch.tensor(70_000.0).half().item())
print("BF16 aralık :", torch.tensor(70_000.0).bfloat16().item(), " (taşmaz ama yuvarlanır)")
print("BF16 hassasiyet: 1 + 0.001 =", (torch.tensor(1.0).bfloat16() + 0.001).item())

---
## 2) Neden hız? LLM üretimi bellek bant genişliğiyle sınırlıdır

Otoregresif üretimde her token için modelin **tüm ağırlıkları** bellekten bir kez okunur, ama her ağırlıkla yalnızca birkaç işlem yapılır (batch = 1 iken matris–vektör çarpımı). GPU'nun hesap birimleri çoğu zaman **veriyi bekler**. Kabaca:

$$\text{token/sn} \;\lesssim\; \frac{\text{bellek bant genişliği (GB/s)}}{\text{model boyutu (GB)}}$$

In [ ]:
gpular = {"Colab T4": 320, "RTX 4090": 1008, "A100 80GB": 2039, "H100 SXM": 3350}
formatlar = {"FP16": 2.0, "INT8": 1.0, "INT4": 0.5}
parametre = 7e9

satirlar = []
for gpu, bant in gpular.items():
    satir = {"GPU (GB/s)": f"{gpu} ({bant})"}
    for fmt, bayt in formatlar.items():
        boyut_gb = parametre * bayt / 1e9
        satir[f"{fmt} ({boyut_gb:.1f} GB)"] = f"~{bant / boyut_gb:.0f} token/sn"
    satirlar.append(satir)
print("7B model için teorik üst sınır (batch = 1):")
pd.DataFrame(satirlar).set_index("GPU (GB/s)")

Bunu kendi makinemizde de görebiliriz: aynı sayıda elemanı kopyalamak (salt bellek trafiği) eleman başına bayt sayısıyla orantılı sürer.

In [ ]:
n = 100_000_000
satirlar = []
for dtype in [torch.float32, torch.float16, torch.int8]:
    x = torch.ones(n, dtype=dtype, device=cihaz)
    x.clone()
    if cihaz == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(5):
        x.clone()
    if cihaz == "cuda":
        torch.cuda.synchronize()
    sure = (time.perf_counter() - t0) / 5
    satirlar.append({"tip": str(dtype), "boyut (MB)": x.element_size() * n / 2**20, "kopyalama (ms)": sure * 1000})
    del x
pd.DataFrame(satirlar).round(1)

---
## 3) Simetrik ve Asimetrik kuantizasyon

### Simetrik (absmax)
Aralık sıfır etrafında simetrik kabul edilir; **zero-point yoktur**.

$$s = \frac{\max|x|}{2^{b-1} - 1}, \qquad q = \text{clamp}\left(\text{round}\left(\frac{x}{s}\right), -(2^{b-1}-1),\; 2^{b-1}-1\right), \qquad \hat x = s \cdot q$$

### Asimetrik (zero-point / affine)
Gerçek $[\min, \max]$ aralığı tüm tam sayı aralığına eşlenir; **sıfır noktası** $z$ kaydırma sağlar.

$$s = \frac{\max(x) - \min(x)}{2^b - 1}, \qquad z = \text{round}\left(-\frac{\min(x)}{s}\right), \qquad q = \text{clamp}\left(\text{round}\left(\frac{x}{s}\right) + z,\; 0,\; 2^b - 1\right), \qquad \hat x = s\,(q - z)$$

| | Simetrik | Asimetrik |
|---|---|---|
| Parametreler | Sadece ölçek $s$ | Ölçek $s$ + sıfır noktası $z$ |
| İdeal dağılım | Sıfır etrafında simetrik (**ağırlıklar**) | Kaydırılmış/tek taraflı (**ReLU/GELU sonrası aktivasyonlar**) |
| Hesaplama | Daha basit ve hızlı matris çarpımı | $z$ için ek düzeltme terimi |

In [ ]:
def simetrik_kuantize(x, bit=8):
    q_max = 2 ** (bit - 1) - 1
    s = x.abs().max() / q_max
    q = torch.clamp(torch.round(x / s), -q_max, q_max)
    return q, s

def simetrik_geri(q, s):
    return q * s

def asimetrik_kuantize(x, bit=8):
    q_max = 2 ** bit - 1
    s = (x.max() - x.min()) / q_max
    z = torch.round(-x.min() / s)
    q = torch.clamp(torch.round(x / s) + z, 0, q_max)
    return q, s, z

def asimetrik_geri(q, s, z):
    return s * (q - z)

# Küçük örnek
x = torch.tensor([-1.2, -0.5, 0.0, 0.3, 0.8, 2.5])
q, s = simetrik_kuantize(x, bit=8)
print(f"Orijinal      : {x.tolist()}")
print(f"Simetrik INT8 : q = {q.int().tolist()}, s = {s:.5f}")
print(f"Geri çevrilmiş: {simetrik_geri(q, s).numpy().round(4).tolist()}")
q, s, z = asimetrik_kuantize(x, bit=8)
print(f"\nAsimetrik UINT8: q = {q.int().tolist()}, s = {s:.5f}, z = {int(z)}")
print(f"Geri çevrilmiş : {asimetrik_geri(q, s, z).numpy().round(4).tolist()}")

### Hangi dağılımda hangisi daha iyi?

- **Ağırlıklar** genellikle sıfır merkezli, normal dağılıma yakın.
- **Aktivasyonlar** (GELU/ReLU sonrası) çoğunlukla pozitif ve çarpık. Simetrik kuantizasyon negatif yarıdaki seviyeleri **boşa harcar**.

In [ ]:
agirliklar = torch.randn(100_000) * 0.02
aktivasyonlar = F.gelu(torch.randn(100_000) * 2 + 0.5)

def mse(a, b):
    return ((a - b) ** 2).mean().item()

satirlar = []
for isim, veri in [("Ağırlıklar (simetrik dağılım)", agirliklar), ("GELU aktivasyonları (çarpık)", aktivasyonlar)]:
    for bit in [8, 4]:
        q, s = simetrik_kuantize(veri, bit)
        qa, sa, za = asimetrik_kuantize(veri, bit)
        satirlar.append({"Veri": isim, "bit": bit,
                         "Simetrik MSE": mse(veri, simetrik_geri(q, s)),
                         "Asimetrik MSE": mse(veri, asimetrik_geri(qa, sa, za)),
                         "Kullanılan seviye (sim.)": len(torch.unique(q)),
                         "Kullanılan seviye (asim.)": len(torch.unique(qa))})
display(pd.DataFrame(satirlar))

fig, eksenler = plt.subplots(1, 2, figsize=(15, 4))
for ax, (isim, veri) in zip(eksenler, [("Ağırlıklar", agirliklar), ("GELU aktivasyonları", aktivasyonlar)]):
    q, s = simetrik_kuantize(veri, 4)
    qa, sa, za = asimetrik_kuantize(veri, 4)
    ax.hist(veri.numpy(), bins=200, alpha=0.4, density=True, label="orijinal")
    for seviye in torch.unique(simetrik_geri(q, s)).tolist():
        ax.axvline(seviye, color="tab:red", alpha=0.6, lw=1)
    for seviye in torch.unique(asimetrik_geri(qa, sa, za)).tolist():
        ax.axvline(seviye, color="tab:green", alpha=0.6, lw=1, ls="--")
    ax.set_title(f"{isim} — INT4 seviyeleri (kırmızı = simetrik, yeşil = asimetrik)")
    ax.legend()
plt.tight_layout(); plt.show()

---
## 4) Aykırı değerler (outliers) ve granülarite

Tek bir büyük değer, **tüm tensörün** ölçeğini büyütür; geri kalan değerler birkaç seviyeye sıkışır. LLM'lerde bazı kanallarda 100× büyük aktivasyonlar görülür (Dettmers vd., *LLM.int8()*, 2022).

Çözüm: ölçeği daha **küçük gruplar** için ayrı ayrı hesaplamak.

| Granülarite | Ölçek sayısı | Kullanım |
|---|---|---|
| Per-tensor | 1 | Basit, aykırı değerlere çok hassas |
| Per-channel (satır başına) | çıkış kanalı sayısı | Ağırlık kuantizasyonunda standart |
| Group-wise (ör. 128 eleman) | eleman / 128 | GPTQ, AWQ, GGUF (`Q4_K`), bitsandbytes NF4 |

In [ ]:
def per_kanal_kuantize(W, bit=4):
    q_max = 2 ** (bit - 1) - 1
    s = W.abs().amax(dim=1, keepdim=True) / q_max             # her satır için ayrı ölçek
    return torch.clamp(torch.round(W / s), -q_max, q_max) * s

def grup_kuantize(W, bit=4, grup=64):
    q_max = 2 ** (bit - 1) - 1
    Wg = W.reshape(-1, grup)                                   # her 64 eleman bir grup
    s = Wg.abs().amax(dim=1, keepdim=True) / q_max
    return (torch.clamp(torch.round(Wg / s), -q_max, q_max) * s).reshape(W.shape)

def per_tensor_kuantize(W, bit=4):
    q, s = simetrik_kuantize(W, bit)
    return simetrik_geri(q, s)

W = torch.randn(1024, 1024) * 0.02
W_aykiri = W.clone()
W_aykiri[7, 100] = 2.0                                         # tek bir aykırı değer (~100σ)

satirlar = []
for isim, matris in [("Normal ağırlık", W), ("1 aykırı değer içeren", W_aykiri)]:
    for bit in [8, 4]:
        satirlar.append({"Matris": isim, "bit": bit,
                         "per-tensor MSE": mse(matris, per_tensor_kuantize(matris, bit)),
                         "per-channel MSE": mse(matris, per_kanal_kuantize(matris, bit)),
                         "group-64 MSE": mse(matris, grup_kuantize(matris, bit))})
tablo = pd.DataFrame(satirlar)
tablo.style.format({c: "{:.2e}" for c in tablo.columns if "MSE" in c})

---
## 5) PTQ — Post-Training Quantization

Model **normal şekilde eğitilir**, sonra ağırlıklar kuantize edilir. Ek eğitim gerekmez; sadece aktivasyonlar için küçük bir **kalibrasyon** veri setiyle aralıklar ölçülür.

Pratikte LLM'ler için PTQ yöntemleri: **GPTQ** (Hessian bilgisiyle hatayı telafi ederek katman katman kuantize eder), **AWQ** (önemli kanalları aktivasyon büyüklüğüne göre korur), **SmoothQuant** (aktivasyon aykırılıklarını ağırlıklara taşır), **bitsandbytes NF4**.

Deney: sklearn `digits` üzerinde bir MLP eğitip ağırlıkları farklı bit sayılarına kuantize ediyoruz (*fake quantization*: kuantize → geri çevir, böylece etkisini FP32 donanımda ölçebiliriz).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

rakamlar = load_digits()
X = torch.tensor(rakamlar.data / 16.0, dtype=torch.float32)
y = torch.tensor(rakamlar.target)
X_egt, X_tst, y_egt, y_tst = [t.to(cihaz) for t in train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)]

def mlp():
    return nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                         nn.Linear(256, 10)).to(cihaz)

def dogruluk(model, X=None, y=None):
    X = X_tst if X is None else X
    y = y_tst if y is None else y
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(-1) == y).float().mean().item()

def egit(model, epok=60, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epok):
        model.train()
        perm = torch.randperm(len(X_egt), device=cihaz)
        for i in range(0, len(X_egt), 64):
            idx = perm[i:i + 64]
            kayip = F.cross_entropy(model(X_egt[idx]), y_egt[idx])
            opt.zero_grad(); kayip.backward(); opt.step()
    return model

torch.manual_seed(0)
fp32_model = egit(mlp())
print(f"FP32 model doğruluğu: %{dogruluk(fp32_model) * 100:.2f}")

In [ ]:
def ptq_agirlik(model, bit, yontem="per-channel"):
    kopya = copy.deepcopy(model)
    with torch.no_grad():
        for katman in kopya:
            if isinstance(katman, nn.Linear):
                if yontem == "per-channel":
                    katman.weight.copy_(per_kanal_kuantize(katman.weight, bit))
                else:
                    katman.weight.copy_(per_tensor_kuantize(katman.weight, bit))
    return kopya

satirlar = []
for bit in [8, 6, 4, 3, 2]:
    satirlar.append({"bit": bit,
                     "per-tensor PTQ": dogruluk(ptq_agirlik(fp32_model, bit, "per-tensor")) * 100,
                     "per-channel PTQ": dogruluk(ptq_agirlik(fp32_model, bit, "per-channel")) * 100})
ptq_tablo = pd.DataFrame(satirlar).set_index("bit")
ptq_tablo.insert(0, "FP32", dogruluk(fp32_model) * 100)
ptq_tablo.round(2)

### Aktivasyon kuantizasyonu ve kalibrasyon

Ağırlıklar sabittir, ama aktivasyonlar girdiye göre değişir. Statik kuantizasyonda aktivasyon aralığı, küçük bir **kalibrasyon** setinde gözlenen min/max ile belirlenir. ReLU sonrası aktivasyonlar ≥ 0 olduğu için burada **asimetrik** kuantizasyon doğal seçimdir.

In [ ]:
class AktivasyonKuantizer(nn.Module):
    def __init__(self, bit=8, simetrik=False):
        super().__init__()
        self.bit, self.simetrik = bit, simetrik
        self.kalibrasyon_modu = True
        self.register_buffer("min_deger", torch.tensor(float("inf")))
        self.register_buffer("max_deger", torch.tensor(float("-inf")))

    def forward(self, x):
        if self.kalibrasyon_modu:                                   # sadece aralığı ölç
            self.min_deger = torch.minimum(self.min_deger, x.min())
            self.max_deger = torch.maximum(self.max_deger, x.max())
            return x
        if self.simetrik:
            q_max = 2 ** (self.bit - 1) - 1
            s = torch.maximum(self.min_deger.abs(), self.max_deger.abs()) / q_max
            return torch.clamp(torch.round(x / s), -q_max, q_max) * s
        q_max = 2 ** self.bit - 1
        s = (self.max_deger - self.min_deger) / q_max
        z = torch.round(-self.min_deger / s)
        return s * (torch.clamp(torch.round(x / s) + z, 0, q_max) - z)


def tam_kuantize_model(model, bit_w, bit_a, simetrik_akt):
    katmanlar = []
    for katman in ptq_agirlik(model, bit_w):
        katmanlar.append(katman)
        if isinstance(katman, nn.ReLU):
            katmanlar.append(AktivasyonKuantizer(bit_a, simetrik_akt).to(cihaz))
    q_model = nn.Sequential(*katmanlar)
    with torch.no_grad():
        q_model(X_egt[:200])                                        # kalibrasyon: 200 örnek
    for m in q_model.modules():
        if isinstance(m, AktivasyonKuantizer):
            m.kalibrasyon_modu = False
    return q_model

satirlar = []
for bit_w, bit_a in [(8, 8), (4, 8), (4, 4), (4, 3)]:
    satirlar.append({"W bit / A bit": f"W{bit_w}A{bit_a}",
                     "simetrik aktivasyon": dogruluk(tam_kuantize_model(fp32_model, bit_w, bit_a, True)) * 100,
                     "asimetrik aktivasyon": dogruluk(tam_kuantize_model(fp32_model, bit_w, bit_a, False)) * 100})
pd.DataFrame(satirlar).set_index("W bit / A bit").round(2)

---
## 6) QAT — Quantization-Aware Training

PTQ çok düşük bit sayılarında (≤ 3 bit) çöker çünkü model bu kadar kaba ağırlıklarla çalışmaya hiç alışmamıştır. QAT, **eğitim sırasında** ileri geçişte kuantize edilmiş ağırlıkları kullanır; model kuantizasyon hatasına dayanıklı olmayı öğrenir.

**Problem:** `round()` fonksiyonunun türevi neredeyse her yerde 0'dır, gradyan akmaz.

**Çözüm – Straight-Through Estimator (STE):** İleri geçişte kuantize et, geri geçişte kuantizasyonu **yokmuş gibi** davran (türevi 1 kabul et):

$$\hat w = w + \text{sg}\big(Q(w) - w\big) \quad\Rightarrow\quad \text{ileri: } \hat w = Q(w), \qquad \text{geri: } \frac{\partial \hat w}{\partial w} = 1$$

($\text{sg}$ = stop-gradient = PyTorch'ta `.detach()`)

In [ ]:
class QATLinear(nn.Linear):
    def __init__(self, in_f, out_f, bit=4):
        super().__init__(in_f, out_f)
        self.bit = bit

    def forward(self, x):
        w_q = per_kanal_kuantize(self.weight, self.bit)
        w_ste = self.weight + (w_q - self.weight).detach()          # STE: ileri Q(w), geri kimlik
        return F.linear(x, w_ste, self.bias)


def qat_modeline_cevir(model, bit):
    katmanlar = []
    for katman in model:
        if isinstance(katman, nn.Linear):
            yeni = QATLinear(katman.in_features, katman.out_features, bit).to(cihaz)
            yeni.load_state_dict(katman.state_dict())               # FP32 ağırlıklardan başla
            katmanlar.append(yeni)
        else:
            katmanlar.append(katman)
    return nn.Sequential(*katmanlar)

# STE'nin gradyan akışını doğrulayalım
w = torch.randn(4, 3, requires_grad=True)
(w + (per_kanal_kuantize(w, 2) - w).detach()).sum().backward()
print("STE gradyanı (hepsi 1 olmalı):\n", w.grad)

In [ ]:
satirlar = []
for bit in [4, 3, 2]:
    ptq = dogruluk(ptq_agirlik(fp32_model, bit)) * 100
    torch.manual_seed(0)
    qat = egit(qat_modeline_cevir(fp32_model, bit), epok=30, lr=5e-4)
    satirlar.append({"bit": bit, "FP32": dogruluk(fp32_model) * 100, "PTQ": ptq, "QAT": dogruluk(qat) * 100})

qat_tablo = pd.DataFrame(satirlar).set_index("bit")
ax = qat_tablo.plot.bar(figsize=(9, 4), rot=0, ylim=(0, 105))
ax.set_ylabel("test doğruluğu (%)")
ax.set_title("Düşük bit sayısında PTQ çöker, QAT toparlar")
for konteyner in ax.containers:
    ax.bar_label(konteyner, fmt="%.1f", fontsize=8)
plt.show()
qat_tablo.round(2)

### Gerçek LLM'leri 4-bit yüklemek (bitsandbytes)

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

ayar = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # normal dağılıma göre seçilmiş 16 seviye
    bnb_4bit_use_double_quant=True,         # ölçek değerlerini de kuantize et
    bnb_4bit_compute_dtype=torch.bfloat16,  # hesaplama BF16'da, depolama 4-bit
)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", quantization_config=ayar, device_map="auto")
print(f"{model.get_memory_footprint() / 2**30:.1f} GB")   # FP16'da ~15 GB → 4-bit ile ~5 GB
```

Diğer yaygın formatlar: **GGUF** (llama.cpp / Ollama, CPU + GPU), **GPTQ** ve **AWQ** (önceden kuantize edilmiş Hub modelleri), **FP8** (H100 ve sonrası).

---
## Özet

| Kavram | Özet |
|---|---|
| FP16 → INT8 → INT4 | Her adımda bellek yarıya iner; üretim bant genişliği sınırlı olduğu için hız da artar |
| Simetrik | $q = \text{round}(x/s)$, zero-point yok; sıfır merkezli **ağırlıklar** için ideal |
| Asimetrik | $q = \text{round}(x/s) + z$; tek taraflı **aktivasyonlar** için seviyeleri boşa harcamaz |
| Granülarite | Per-tensor < per-channel < group-wise; aykırı değerlere karşı küçük gruplar |
| **PTQ** | Eğitimden sonra, hızlı ve ucuz; 8-bit neredeyse kayıpsız, 4-bit iyi yöntemle (GPTQ/AWQ) iyi, ≤3 bit zor |
| **QAT** | Eğitim sırasında fake-quant + STE; çok düşük bitte bile doğruluğu korur ama eğitim maliyeti var |